In [12]:
import pandas as pd
listings_clean = pd.read_csv('../data/listings_clean_outlier.csv', parse_dates=['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review'])
listings = listings_clean.copy()

In [13]:
listings.shape

(4574, 74)

# 1)

In [14]:
print(listings.review_scores_rating.mean().round(2),
      listings.review_scores_rating.median(),
      listings.review_scores_rating.std().round(2))

4.82 4.92 0.32


A large number of reviews are exactly 5 with a few outliers being less than 5.
I'd use the mean most likely, the mean and median say different thing, all depends what you are trying to say.

# 2)

In [17]:
q1 = listings.review_scores_rating.quantile(0.25)
q3 = listings.review_scores_rating.quantile(0.75)
iqr = q3 - q1
print(listings.review_scores_rating.describe().round(0))
print(round(q1, 1), round(q3, 1), round(iqr, 1))

count    3930.0
mean        5.0
std         0.0
min         1.0
25%         5.0
50%         5.0
75%         5.0
max         5.0
Name: review_scores_rating, dtype: float64
4.8 5.0 0.2


In [18]:
upper_fence = q3 + 1.5 * iqr
lower_fence = q1 - 1.5 * iqr
print(round(lower_fence, 1), round(upper_fence, 1))

4.4 5.4


In [19]:
outliers = listings[listings.review_scores_rating < lower_fence]
print(len(outliers))

211


Keep them, ratings below the lower fence are very much valid. We have no reason to suspect anything else. Also, there's no reason to mark them.

# 3 a)

In [20]:
listings.groupby('host_is_superhost').agg(
    num=('number_of_reviews', 'count'),
    median_num_reviews=('number_of_reviews', 'median')
)

,num,median_num_reviews
host_is_superhost,,
False,3837,7.0
True,737,53.0


# b)

In [21]:
listings['years_existed'] = (listings.last_scraped - listings.first_review).dt.days / 365.25

In [24]:
listings.groupby('host_is_superhost').agg(
    num=('years_existed', 'count'),
    median_years_existed=('years_existed', 'median')
).round(2)

,num,median_years_existed
host_is_superhost,,
False,3215,2.89
True,715,3.31


There's likely a connection between the median years and the superhost. But we can't be sure what the connection is.

# c)

In [26]:
listings.groupby('host_is_superhost').agg(
    num=('number_of_reviews_ltm', 'count'),
    median_reviews_last_12_months=('number_of_reviews_ltm', 'median')
).round(2)

,num,median_reviews_last_12_months
host_is_superhost,,
False,3837,1.0
True,737,16.0


In the last 12 months, non-superhosts have a low median of received reviews. But we can't conclude much from a median alone, it's just a median and we would need to look at different values like mean and std to draw any solid conclusions.

To be a superhost you have to meet different criteria. Hosts who have no reviews are unlikely to be superhosts for this reason and it skews the median for the two groups to the point that no further conclusions can really be made.

# 4)

Keep everything, mark nothing.
Why? we have no reason to believe any of the data is invalid. We have no reason the exclude them from any data analysis.

# 5)

How many NaN values does the whole dataframe have and how many percent is that?

In [31]:
total_isna = 0
total_values = 0
for col in listings.columns:
    total_isna += listings[col].isna().sum()
    total_values += listings[col].count()
print(total_isna, round(100*total_isna/total_values, 2))

21971 6.84
